# 02 - Data Cleaning and Preparation

## Electricity Consumption Analysis and Short-Term Demand Forecasting

### Objective

Clean and prepare the raw electricity consumption dataset for exploratory data analysis and machine learning.

The main tasks in this notebook are:

1. Combine `Date` and `Time` into a single datetime column.
2. Convert measurement columns to numeric data types.
3. Identify and handle missing measurements.
4. Check for duplicate records.
5. Sort observations chronologically.
6. Create a clean dataset for the next stage of the project.
7. Save the cleaned data in the `data/processed/` folder.

### Important

The original raw dataset in `data/raw/` will not be modified.


## 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load the Raw Dataset

We reload the original dataset so that this notebook starts directly from the untouched raw data.


In [2]:
file_path = "../data/raw/household_power_consumption.txt"

df = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    low_memory=False
)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 3. Inspect Missing Values

The initial inspection showed 25,979 missing observations in each of the seven measurement columns.

We will investigate the missing data before deciding how to handle it.


In [3]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary


,Missing Values,Missing Percentage
Date,0,0.00
Time,0,0.00
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25


## 4. Combine Date and Time

The original dataset stores the date and time separately.

For time-series analysis, it is more useful to have one datetime column. We create `DateTime` using the `Date` and `Time` columns.


In [4]:
df["DateTime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True,
    errors="coerce"
)

df[["Date", "Time", "DateTime"]].head()


,Date,Time,DateTime
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00


## 5. Check DateTime Conversion

Check whether any values failed to convert into valid datetime values.


In [5]:
print("Invalid DateTime values:", df["DateTime"].isna().sum())
print("First DateTime:", df["DateTime"].min())
print("Last DateTime:", df["DateTime"].max())


Invalid DateTime values: 0
First DateTime: 2006-12-16 17:24:00
Last DateTime: 2010-11-26 21:02:00


## 6. Convert Measurement Columns to Numeric

The electricity measurements should be numeric so that mathematical calculations, aggregation, visualization, and machine-learning algorithms can use them correctly.

We use `pd.to_numeric()` with `errors="coerce"` so invalid values become `NaN` instead of causing the notebook to fail.


In [6]:
measurement_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for column in measurement_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[measurement_columns].dtypes


Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

## 7. Check Missing Values After Conversion

Recalculate missing values after converting the measurement columns to numeric.


In [7]:
df.isnull().sum()


Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
DateTime                     0
dtype: int64

## 8. Check for Duplicate Records

Because this is time-series data, duplicate rows should be identified before further analysis.


In [8]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)


Duplicate rows: 0


## 9. Remove Duplicate Records

If duplicate rows exist, remove them while keeping the first occurrence.

If the result above is zero, this operation will not change the dataset.


In [9]:
df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)


Shape after removing duplicates: (2075259, 10)


## 10. Sort Data Chronologically

Time-series observations must be in chronological order.

We sort the data using the newly created `DateTime` column.


In [10]:
df = df.sort_values("DateTime").reset_index(drop=True)

df[["DateTime", "Global_active_power"]].head()


,DateTime,Global_active_power
0,2006-12-16 17:24:00,4.216
1,2006-12-16 17:25:00,5.360
2,2006-12-16 17:26:00,5.374
3,2006-12-16 17:27:00,5.388
4,2006-12-16 17:28:00,3.666


## 11. Check the Time Intervals

The raw dataset is expected to contain measurements at one-minute intervals.

We calculate the difference between consecutive timestamps to identify whether the time series contains gaps.


In [11]:
time_difference = df["DateTime"].diff().dropna()

time_difference.value_counts().head(10)


DateTime
0 days 00:01:00    2075258
Name: count, dtype: int64

### Interpretation

A one-minute interval should appear most frequently.

Longer intervals indicate periods where observations are missing. We will preserve the time-series structure and handle missing measurements before aggregation.


## 12. Handle Missing Measurement Values

The dataset contains missing measurements.

For time-series electricity data, simply replacing missing values with zero would be incorrect because zero means no electricity consumption, whereas a missing value means the measurement was not recorded.

We therefore use **time-based interpolation** for the measurement columns.

This estimates a missing value using nearby observations in time.


In [12]:
df = df.set_index("DateTime")

df[measurement_columns] = df[measurement_columns].interpolate(
    method="time",
    limit_direction="both"
)

df = df.reset_index()


## 13. Verify Missing Values After Interpolation


In [13]:
remaining_missing = df[measurement_columns].isnull().sum()
remaining_missing


Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

### Important Data-Cleaning Decision

Time-based interpolation is used here because the project is focused on continuous electricity-consumption patterns.

For the final report, we will document:

> Missing measurement values were identified in the raw dataset and handled using time-based interpolation. Missing values were not replaced with zero because an unrecorded measurement does not imply zero electricity consumption.


## 14. Remove Original Date and Time Columns

The combined `DateTime` column now contains the information from both `Date` and `Time`.

We keep `DateTime` and remove the original text columns to create a cleaner dataset.


In [14]:
df = df.drop(columns=["Date", "Time"])

df.head()


,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 15. Verify Final Data Types


In [15]:
df.dtypes


DateTime                 datetime64[us]
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
dtype: object

## 16. Final Dataset Information


In [16]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   DateTime               datetime64[us]
 1   Global_active_power    float64       
 2   Global_reactive_power  float64       
 3   Voltage                float64       
 4   Global_intensity       float64       
 5   Sub_metering_1         float64       
 6   Sub_metering_2         float64       
 7   Sub_metering_3         float64       
dtypes: datetime64[us](1), float64(7)
memory usage: 126.7 MB


## 17. Final Dataset Summary


In [17]:
print("Final dataset shape:", df.shape)
print("Start:", df["DateTime"].min())
print("End:", df["DateTime"].max())
print("Remaining missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


Final dataset shape: (2075259, 8)
Start: 2006-12-16 17:24:00
End: 2010-11-26 21:02:00
Remaining missing values: 0
Duplicate rows: 0


## 18. Save the Cleaned Dataset

The raw dataset is kept unchanged.

The cleaned dataset is saved in `data/processed/` and will be used in the next stage of the project.


In [18]:
output_path = "../data/processed/cleaned_household_power_consumption.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:")
print(output_path)


Cleaned dataset saved to:
../data/processed/cleaned_household_power_consumption.csv


## 19. Final Preview

Display a few rows of the cleaned dataset.


In [19]:
df.head()

,DateTime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Conclusion

The raw electricity-consumption data has been prepared for further analysis.

### Completed tasks

- Loaded the raw dataset.
- Combined `Date` and `Time` into `DateTime`.
- Converted measurement columns to numeric.
- Checked for duplicate records.
- Sorted observations chronologically.
- Investigated time intervals.
- Handled missing measurement values using time-based interpolation.
- Removed redundant original date/time columns.
- Verified the cleaned dataset.
- Saved the cleaned dataset for the next stage.

### Next notebook

The next stage will be:

**03 - Hourly Aggregation**

The original data contains minute-level observations. We will aggregate the cleaned data into hourly electricity consumption so that the subsequent EDA and forecasting models are practical and easier to interpret.
